# Day 075 — Exercise 1: generate_speech

**What you'll build:** `generate_speech(text, voice, rate, pitch, tts_fn=None) -> bytes` — synthesise speech audio from text using edge-tts.

**Why it matters:** Stage 1 of the talking-head pipeline — turns the lesson script into the audio track that drives everything else.

In [ ]:
from pathlib import Path
_mock_tts_fn     = lambda text, voice, rate, pitch: b'MP3:' + text[:8].encode()


## Task

- **Mock:** `if tts_fn is not None: return tts_fn(text, voice, rate, pitch)`
- **Real:** `import asyncio, edge_tts`; define `async def _run()` that creates `edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)`, streams chunks, collects `chunk['data']` where `chunk['type']=='audio'`, returns `b''.join(chunks)`
- Return `asyncio.run(_run())`

## Your Implementation

In [ ]:
def generate_speech(text: str, voice: str = 'en-US-AriaNeural',
                    rate: str = '+0%', pitch: str = '+0Hz',
                    tts_fn=None) -> bytes:
    """Generate speech audio bytes from text.

    Args:
        text:   text to synthesise
        voice:  edge-tts ShortName
        rate:   speaking rate adjustment ('+10%', '-5%', ...)
        pitch:  pitch adjustment ('+5Hz', '-10Hz', ...)
        tts_fn: callable(text, voice, rate, pitch) -> bytes for testing
    """
    raise NotImplementedError


In [ ]:
def generate_speech(text, voice='en-US-AriaNeural',
                    rate='+0%', pitch='+0Hz', tts_fn=None):
    if tts_fn is not None:
        return tts_fn(text, voice, rate, pitch)
    import asyncio, edge_tts
    async def _run():
        comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
        chunks = []
        async for chunk in comm.stream():
            if chunk['type'] == 'audio':
                chunks.append(chunk['data'])
        return b''.join(chunks)
    return asyncio.run(_run())


## Automated checks

In [ ]:

score, total = 0, 5
try:
    # returns bytes
    audio = generate_speech('Hello!', tts_fn=_mock_tts_fn)
    assert isinstance(audio, bytes)
    score += 1; print("✅ returns bytes")

    # non-empty output
    assert len(audio) > 0
    score += 1; print("✅ output is non-empty")

    # tts_fn receives all 4 args
    captured = {}
    def _cap(text, voice, rate, pitch):
        captured.update(text=text, voice=voice, rate=rate, pitch=pitch)
        return b'OK'
    generate_speech('Hi', voice='en-GB-LibbyNeural', rate='+10%', pitch='+5Hz',
                    tts_fn=_cap)
    assert captured.get('text') == 'Hi'
    assert captured.get('voice') == 'en-GB-LibbyNeural'
    score += 1; print("✅ tts_fn receives text and voice correctly")

    # rate and pitch forwarded
    assert captured.get('rate') == '+10%' and captured.get('pitch') == '+5Hz'
    score += 1; print("✅ rate and pitch forwarded to tts_fn")

    # different texts → different audio
    a1 = generate_speech('A', tts_fn=_mock_tts_fn)
    a2 = generate_speech('BCDEFGHIJ', tts_fn=_mock_tts_fn)
    assert a1 != a2
    score += 1; print("✅ different texts produce different audio bytes")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def generate_speech(text, voice='en-US-AriaNeural',
                    rate='+0%', pitch='+0Hz', tts_fn=None):
    if tts_fn is not None:
        return tts_fn(text, voice, rate, pitch)
    import asyncio, edge_tts
    async def _run():
        comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
        chunks = []
        async for chunk in comm.stream():
            if chunk['type'] == 'audio':
                chunks.append(chunk['data'])
        return b''.join(chunks)
    return asyncio.run(_run())
```

**Why pass all 4 args to tts_fn?** The mock receives `(text, voice, rate, pitch)` so it can be a drop-in for the real function. A mock that only uses `text` still accepts the other args — this makes the interface symmetric.

</details>